# 06 · Agentic Design, Dialogue State & System Integration

**Objective.** Add a ReAct-style agent with tool use and dialogue-state tracking, then integrate every component into one end-to-end pipeline with a UI.

**Advanced techniques covered:** Agentic design, **ReAct** prompting, dialogue state tracking, automated **escalation** as an LLM decision; **system integration** of all basic + advanced techniques.

>  Logic lives in `src/`; these notebooks orchestrate, experiment, visualise and analyse (good-practice separation, ULO6). Run the notebooks in order `01 -> 07` after completing setup in `README.md`.

In [1]:
# --- bootstrap: make `import config` and `from src import ...` work from anywhere ---
import sys, os
from pathlib import Path
p = Path.cwd()
ROOT = next((c for c in [p, *p.parents] if (c/'config.py').exists() and (c/'src').exists()), p)
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print('project root:', ROOT)

project root: C:\Users\lenovo\Desktop\ANLP_8420_GROUPC\Assignment3_GroupC\Codes


## 1. Agentic Reasoning and Tool Use

Unlike a standard prompt-response system, the agentic workflow can perform multiple reasoning and retrieval steps before generating a final response.

The agent first analyses the customer request, identifies the required actions, retrieves relevant information from the knowledge base, and then decides whether the issue can be resolved automatically or should be escalated.

The reasoning trace shown below illustrates the sequence of decisions made by the agent when handling a delayed order enquiry.

In [2]:
from src.rag import RAGStore
from src.agent import ReActAgent, DialogueState
store = RAGStore()
agent = ReActAgent(rag_store=store, max_steps=6)
out = agent.run('Where is my order 48213? It has been ages.')
import json
print(json.dumps(out['trace'], indent=2, default=str))
print('\nFINAL:', out['final_answer'])
print('state:', out['state'].summary())

C:\Users\lenovo\anaconda3\envs\comp8420\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[
  {
    "thought": "Get order status for order 48213",
    "action": "get_order_status",
    "action_input": "48213"
  },
  {
    "thought": "Order is in transit and was dispatched 14 business days ago. Check if this aligns with expected delivery times.",
    "action": "search_policy",
    "action_input": "expected delivery time for ParcelCo"
  },
  {
    "thought": "The customer requested the status of their order 48213, which was dispatched 14 business days ago via ParcelCo. The current status is in transit. According to our policy, orders delayed beyond 14 business days may be cancelled for a full refund of the item price.",
    "action": "search_policy",
    "action_input": "order delayed beyond 14 business days cancellation policy"
  },
  {
    "thought": "The customer requested the status of their order 48213, which was dispatched 14 business days ago via ParcelCo. The current status is in transit. According to our policy, orders delayed beyond 14 business days may be cancelled

## 2. Dialogue State Tracking Across Turns

Customer support conversations often span multiple messages. To maintain context, the system stores information gathered from previous interactions, including extracted entities, conversation history, and resolution status.

The dialogue state enables the system to handle follow-up questions without requiring the customer to repeat information that has already been provided.

In [3]:
state = DialogueState()
agent.run('Hi, I have a problem with order 48213.', state)
agent.run('It still has not arrived, can I get a refund?', state)
print(state.summary())
for t in state.turns:
    print(f"  [{t['role']}] {t['text'][:80]}")

category=None; slots: none; resolved=False; escalated=True; turns=2
  [customer] Hi, I have a problem with order 48213.
  [customer] It still has not arrived, can I get a refund?


## 3. End-to-End Pipeline Integration

This experiment demonstrates how the individual NLP components developed in earlier notebooks operate together within a single workflow.

For each customer enquiry, the system performs:
1. Intent classification
2. Sentiment analysis
3. Named entity recognition
4. Knowledge retrieval
5. Response generation
6. Escalation decision making

The resulting output contains both the intermediate NLP results and the final customer-facing response.

In [4]:
from src.pipeline import CustomerServicePipeline
pipe = CustomerServicePipeline(use_rag=True, prompt_variant='few_shot_cot')
res = pipe.process('My order #48213 still hasn\'t arrived after two weeks and nobody is replying. This is unacceptable.')
import json; print(json.dumps(res.to_dict(), indent=2)[:1500])

[pipeline] loading components ...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[pipeline] ready.
{
  "message": "My order #48213 still hasn't arrived after two weeks and nobody is replying. This is unacceptable.",
  "category": "ORDER",
  "category_confidence": 0.3404,
  "sentiment": {
    "label": "negative",
    "score": 0.4588,
    "compound": -0.4588,
    "priority": false,
    "backend": "vader"
  },
  "entities": [
    {
      "text": "order #48213",
      "label": "ORDER_NUMBER",
      "start": 3,
      "end": 15
    },
    {
      "text": "two weeks",
      "label": "DATE",
      "start": 43,
      "end": 52
    }
  ],
  "retrieved": [
    {
      "text": "## Q: want help to chck how soon can i expect my order\nWe recognize your eagerness to receive your order and your interest in tracking its progress to determine its estimated arrival time. To provide you with accurate information, could you please provide us with your Order Number or Tracking Number? With that information, we will be able to give you an update on the status of your order and an estimat

### Auto-resolve vs escalation
A confident, covered request auto-resolves; an ambiguous one trips the escalation gate - demonstrating the system *knows what it doesn't know*.

In [5]:
amb = pipe.process('I have a complicated situation involving a dispute and I am not sure this is even the right place.')
print('category   :', amb.category, amb.category_confidence)
print('escalate   :', amb.should_escalate)
print('reasons    :', amb.escalation_reasons)

category   : DELIVERY 0.2356
escalate   : True
reasons    : ['LLM flagged the request for escalation', 'low classifier confidence (0.24)', 'low response confidence (0.50)']


## 4. User Interface Demonstration

A lightweight Streamlit interface was developed to demonstrate the complete customer service workflow.

The interface allows users to submit customer enquiries and view the resulting classification, sentiment analysis, extracted entities, retrieved knowledge, generated response, and escalation decision.

This provides a practical demonstration of how the system could be used within a real customer support environment.

In [6]:
# Interactive single-pass demo (styled HTML output)
import html as _html
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

box = widgets.Textarea(
    value='My order 48213 is two weeks late and I am really upset about it.',
    placeholder='Type a customer message…',
    layout=widgets.Layout(width='640px', height='80px'))
btn = widgets.Button(description='Analyse message', button_style='primary', icon='play')
out_area = widgets.Output()

def _badge(text, color):
    return (f"<span style='background:{color};color:#fff;padding:2px 9px;"
            f"border-radius:999px;font-size:12px;font-weight:600'>{_html.escape(str(text))}</span>")

def _run(_):
    with out_area:
        clear_output()
        r = pipe.process(box.value)
        sent_col = {'positive':'#2a9d8f','negative':'#d62828','neutral':'#8d99ae'}.get(r.sentiment['label'],'#8d99ae')
        dec = ("<span style='color:#d62828;font-weight:700'> ESCALATE</span>"
               if r.should_escalate else
               "<span style='color:#2a9d8f;font-weight:700'> AUTO-RESOLVE</span>")
        ents = " ".join(_badge(f"{e['label']}:{e['text']}", '#1d3557') for e in r.entities) or "<i>none</i>"
        srcs = ", ".join(c['source'] for c in r.retrieved) or "<i>none</i>"
        reasons = "; ".join(r.escalation_reasons) or "all signals above threshold"
        display(HTML(f"""
        <div style='font-family:system-ui;max-width:760px;border:1px solid #e7ecf2;
                    border-radius:12px;padding:14px 16px'>
          <div style='margin-bottom:8px'>{dec}
            &nbsp; {_badge(r.category, '#2a6f97')} conf {r.category_confidence:.2f}
            &nbsp; {_badge(r.sentiment['label'], sent_col)} ({r.sentiment['compound']:.2f})</div>
          <div style='background:#f1f8f5;border-left:4px solid #2a9d8f;padding:10px 12px;
                      border-radius:8px;margin:8px 0'>{_html.escape(r.response or ' - ')}</div>
          <div style='font-size:13px;color:#333'><b>Entities:</b> {ents}</div>
          <div style='font-size:13px;color:#333;margin-top:4px'><b>Knowledge sources:</b> {srcs}</div>
          <div style='font-size:13px;color:#555;margin-top:4px'><b>Decision reason:</b> {reasons}</div>
        </div>"""))

btn.on_click(_run)
display(box, btn, out_area)

Textarea(value='My order 48213 is two weeks late and I am really upset about it.', layout=Layout(height='80px'…

Button(button_style='primary', description='Analyse message', icon='play', style=ButtonStyle())

Output()

## Analysis

The integrated pipeline demonstrates how multiple NLP and LLM components can work together to support customer service automation.

Traditional NLP techniques provide structured information such as intent labels, sentiment scores, and extracted entities, while retrieval-augmented generation supplies grounded company knowledge. The agentic layer coordinates these components and determines whether the enquiry can be resolved automatically or requires human intervention.

The results show that combining multiple techniques produces more informative and reliable responses than any individual component alone.

## Limitations

Although the agentic workflow improves decision making, its performance depends on the quality of the underlying NLP components and retrieved knowledge.

Classification errors, missing entities, or retrieval failures can affect downstream decisions and response quality. In addition, multi-step reasoning introduces additional computational overhead compared with a simple prompt-response approach.

Future work could explore more advanced planning strategies, tool selection mechanisms, and multi-agent architectures.

## Summary

This notebook integrated the major components of the intelligent customer service system into a single agentic workflow.

The implementation combines:
- Intent classification
- Sentiment analysis
- Named entity recognition
- Retrieval-augmented generation
- Dialogue state tracking
- Escalation decision making
- User interface interaction

The experiments demonstrate how multiple NLP and LLM techniques can be coordinated to automate customer support while maintaining context and providing grounded responses.

**Next:** Notebook 07 evaluates system performance, compares alternative configurations, and measures the impact of retrieval and agentic decision making.